# Stage 3: Generate Parametric Combinations

**Purpose**: Expands parametric templates into all possible concrete item variants through Cartesian product of parameter values.

**Input**: `data/intermediate/OBRA CIVIL/OBRA CIVIL.json`  
**Output**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage3.json`

## What this notebook does

BC3 templates define base construction items with configurable parameters. For example, a single "Concrete-encased PVC conduit" template may have parameters for:

- Number of pipes: 2, 4, 6, 8, 12
- Diameter: 110mm, 160mm, 200mm  
- Terrain type: normal, rocky, under tracks, road crossing, platform, ballast
- Work shift: daytime, nighttime, exceptional conditions

This notebook generates the Cartesian product of all parameter combinations, expanding ~545 base templates into ~127,000 unique item variants.

## Key transformation

| Before | After |
|--------|-------|
| `OEB020$` (template with parameters) | `OEB020$AAAAAA`, `OEB020$AAAAAB`, ... (one per combination) |

Each generated item retains a `parent_key` reference to its source template for hierarchical evaluation.

In [ ]:
import json
import os
import datetime
import time

# Sprint 17 (Task D1): the core Cartesian-expansion transform now lives in
# synthetic.stage_runners as an importable, pure, in-memory function. This
# notebook keeps its own file-IO / logging / statistics driver cells and
# delegates the transform to the extracted runner.
from synthetic.stage_runners import transform_data

# Load the JSON file
# Input: A file path to a JSON file.
# Output: The loaded JSON object.
def load_json(file_path):
    """Load and return the JSON content from the given file."""
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

# Save the transformed data to a JSON file
# Input: Transformed JSON data and the output file path.
# Output: None (Saves the JSON data to the specified file).
def save_json(data, output_path):
    """Save the transformed data to a file in JSON format."""
    with open(output_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

# Generate a structured filename
# Input: Source file path, stage number, output directory, and file extension.
# Output: A structured file name for the output file.
def generate_filename(input_file, stage, output_dir, extension="json"):
    """Generate a structured filename with stage and timestamp."""
    base_name = os.path.splitext(os.path.basename(input_file))[0]  # Get base name of the source file
    file_name = f"{base_name}_stage{stage}.{extension}"
    return os.path.join(output_dir, file_name)

# Create output directories dynamically within the source file's path
# Input: Source file path.
# Output: Path to the created output directory.
def create_output_dirs(input_file):
    """Create and return the output directory path within the source file's directory."""
    input_dir = os.path.dirname(input_file)
    output_dir = input_dir # Same directory
    os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists
    return output_dir

# Log process details
# Input: Source file path and a dictionary of stage outputs.
# Output: None (Writes the log to a file).
def log_process(input_file, stage_outputs):
    """Log the details of the process and output files."""
    log_file = os.path.join(os.path.dirname(input_file), "process_log.txt")
    with open(log_file, "a") as log:
        log.write(f"Source File: {input_file}\n")
        for stage, output in stage_outputs.items():
            log.write(f"Stage {stage} Output: {output}\n")
        log.write("\n")

# Calculate statistics about the transformation process
# Input: Original data and transformed data.
# Output: A dictionary containing statistics.
def calculate_statistics(original_data, transformed_data):
    """Calculate statistics about the transformation process."""
    stats = {
        "total_items_original": len(original_data),
        "total_items_transformed": len(transformed_data),
        "parameters_per_item": {},
        "values_per_parameter": {},
    }

    for parent_key, content in original_data.items():
        if "parameters" in content:
            params = content["parameters"]
            num_params = len(params)
            values_per_param = {key: len(param["values"]) for key, param in params.items()}

            stats["parameters_per_item"][parent_key] = num_params
            stats["values_per_parameter"][parent_key] = values_per_param

    return stats

# Run the file
def run_s03_generate_parametric_combinations(input_file):
    output_dir = create_output_dirs(input_file)  # Create the output directory

    try:
        # Start the timer
        start_time = time.time()

        # Stage 2: Transform data
        data = load_json(input_file)  # Load the source JSON file
        stage3_output = generate_filename(input_file, stage=3, output_dir=output_dir)  # Generate output file name
        transformed_data = transform_data(data)  # Transform the data
        save_json(transformed_data, stage3_output)  # Save the transformed data
        print(f"Stage 3 output saved to {stage3_output}")

        # Calculate and display statistics
        stats = calculate_statistics(data, transformed_data)
        print("Transformation Statistics:")
        print(f"  Total Items in Original Data: {stats['total_items_original']}")
        print(f"  Total Items in Transformed Data: {stats['total_items_transformed']}")

        # End the timer and display runtime
        end_time = time.time()
        print(f"Process completed in {end_time - start_time:.2f} seconds.")

        # Log the output details
        stage_outputs = {3: stage3_output}
        log_process(input_file, stage_outputs)

    except Exception as e:
        print(f"Error: {e}")
   
# Input Format:
# {
#     "<parent_key>$": {
#         "ud": "...",
#         "concept": "...",
#         "text_variables": {...},
#         "resumen": "...",
#         "texto": "...",
#         "parameters": {
#             "<parameter_name>": {
#                 "label": "...",
#                 "values": [
#                     {"label": "...", "value": "..."},
#                     ...
#                 ]
#             }
#         }
#     },
#     ...
# }

# Output Format:
# {
#     "<transformed_key>": {
#         "parent_key": "<original_parent_key>",
#         "ud": "...",
#         "concept": "...",
#         "text_variables": {...},
#         "resumen": "...",
#         "texto": "...",
#         "parameters": {
#             "<parameter_name>": {
#                 "label": "...",
#                 "values": [
#                     {"label": "...", "value": "..."}
#                 ]
#             }
#         }
#     },
#     ...
# }

In [16]:
from utils import config

input_file = config.chapter_path("OBRA CIVIL")  # change chapter name as needed
run_s03_generate_parametric_combinations(input_file) 

Stage 3 output saved to /work/data/intermediate/OBRA CIVIL/OBRA CIVIL_stage3.json
Transformation Statistics:
  Total Items in Original Data: 545
  Total Items in Transformed Data: 126938
Process completed in 12.25 seconds.
